In [1]:
# Google Colab / local setup
import os, sys, subprocess
from pathlib import Path
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    CODE_ROOT = Path("/content/drive/MyDrive/msc project/code")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e",
        str(CODE_ROOT), 'catboost==1.2.10', 'rapidfuzz==3.14.3',
    ])
else:
    CODE_ROOT = next(
        path for path in (Path.cwd(), *Path.cwd().parents)
        if (path / "pyproject.toml").exists()
    )
os.chdir(CODE_ROOT)
sys.path.insert(0, str(CODE_ROOT))
CODE = CODE_ROOT

# 06b - VIX admission

## Methodology

The VIX experiment pairs price-only and price+VIX M15 inputs across five embargoed 2024 OOF folds, nine model families and three label widths. The simple majority gate tests whether VIX improves net economics consistently before H1 2025 or forward evidence is examined.

USA500 and USATECH are separate. Every economic table reports Net, Sharpe,
Sortino and drawdown after a fixed all-in round-trip deduction: 2 bps (0.02%)
for USA500 and 3 bps (0.03%) for USATECH. This covers bid-ask spread plus
commission/slippage; bar-specific spreads remain diagnostic, not additional
costs. Inputs stop before `2026-04-01 00:00 UTC`; Q2-2026 is evaluated separately in Notebook 07 and
no model is refitted here.

In [2]:
import json
import numpy as np
import pandas as pd
from IPython.display import display
from experiments.build_notebook_06 import MODEL_LABELS

STREAMS = ("usa500", "usatech")
ROOT = CODE_ROOT / "experiments" / "cache" / "index_replication"
summary_rows, effect_rows = [], []
for stream in STREAMS:
    decision = json.loads((ROOT / stream / "vix_admission.json").read_text(encoding="utf-8"))
    paired = pd.read_parquet(ROOT / stream / "vix_gate_paired_2024.parquet")
    assert decision["gate_complete"] and decision["frozen_before_sentiment"]
    assert decision["conditions"]["families_positive_majority_5_of_9"]
    summary_rows.append({
        "Index": stream.upper(),
        "Selected input": "Price + VIX" if decision["selected_base"] == "price_vix" else "Price only",
        "Median gate net delta (pp)": 100 * float(decision["metrics"]["median_family_delta"]),
        "Positive models / 9": int(decision["metrics"]["positive_families"]),
        "Positive folds / 5": int(decision["metrics"]["positive_folds"]),
        "Trade retention %": 100 * float(decision["metrics"]["trade_retention"]),
    })
    model_width = paired.groupby(["model_name", "width_bps"], as_index=False).agg(
        price_net=("price_net", "sum"),
        vix_net=("vix_net", "sum"),
        price_trades=("price_trades", "sum"),
        vix_trades=("vix_trades", "sum"),
    )
    model_width["net_delta"] = model_width["vix_net"] - model_width["price_net"]
    representative = model_width.sort_values(
        ["model_name", "net_delta", "width_bps"]
    ).groupby("model_name", as_index=False).nth(1).set_index("model_name")
    for row in decision["family_deltas"]:
        model = row["model_name"]
        pair = representative.loc[model]
        assert abs(float(pair["net_delta"]) - float(row["net_delta"])) < 1e-12
        price_trades = int(pair["price_trades"])
        vix_trades = int(pair["vix_trades"])
        effect_rows.append({
            "Index": stream.upper(),
            "Model": MODEL_LABELS[model],
            "Gate width bps": int(pair["width_bps"]),
            "Price-only gate net %": 100 * float(pair["price_net"]),
            "Price + VIX gate net %": 100 * float(pair["vix_net"]),
            "Gate net delta (pp)": 100 * float(row["net_delta"]),
            "Price trades": price_trades,
            "Price + VIX trades": vix_trades,
            "Trade retention %": 100 * vix_trades / price_trades if price_trades else 0.0,
        })
summary = pd.DataFrame(summary_rows).sort_values(
    ["Median gate net delta (pp)", "Index"], ascending=[False, True]
)
effects = pd.DataFrame(effect_rows).sort_values(
    ["Gate net delta (pp)", "Index", "Model"], ascending=[False, True, True]
)

## Admission decision

Method: The 2024 gate selects VIX when the median net delta is positive, at least 5/9 model families and 3/5 folds are positive, and aggregate trade retention is at least 80%.

In [3]:
# non-economic-table
display(summary.round({"Median gate net delta (pp)": 3, "Trade retention %": 1}))
print("Takeaway: VIX improves relative Net for 6/9 USA500 models and 5/9 USATECH models, although the representative gate strategies remain unprofitable.")

,Index,Selected input,Median gate net delta (pp),Positive models / 9,Positive folds / 5,Trade retention %
1,USATECH,Price + VIX,4.366,5,4,103.0
0,USA500,Price + VIX,1.970,6,3,103.6


Takeaway: VIX improves relative Net for 6/9 USA500 models and 5/9 USATECH models, although the representative gate strategies remain unprofitable.


## Numeric model-family effects

Method: Five fold results are summed within each width; the middle of the three width-level net deltas is the family vote, and the table shows that exact price/VIX pair and its aggregate gate trades.

In [4]:
# vix-numeric-table
# non-economic-table
display(effects.round({"Price-only gate net %": 3, "Price + VIX gate net %": 3, "Gate net delta (pp)": 3, "Trade retention %": 1}))
print("Takeaway: VIX often reduces the loss relative to price-only, but every representative absolute gate return remains negative.")

,Index,Model,Gate width bps,Price-only gate net %,Price + VIX gate net %,Gate net delta (pp),Price trades,Price + VIX trades,Trade retention %
15,USATECH,MLP,10,-67.579,-49.455,18.124,2051,2160,105.3
16,USATECH,LSTM,10,-68.047,-55.996,12.051,1999,2174,108.8
9,USATECH,LogReg,10,-49.898,-38.721,11.177,1896,2039,107.5
12,USATECH,Linear SVM,5,-28.210,-23.759,4.451,901,943,104.7
10,USATECH,Decision Tree,10,-75.298,-70.932,4.366,2219,2159,97.3
8,USA500,GRU,15,-31.963,-28.429,3.534,1523,1555,102.1
5,USA500,CatBoost,10,-26.190,-23.070,3.121,999,930,93.1
7,USA500,LSTM,15,-31.444,-28.334,3.110,1417,1549,109.3
0,USA500,LogReg,15,-24.370,-21.529,2.841,1392,1562,112.2
3,USA500,Linear SVM,5,-13.310,-11.339,1.970,465,531,114.2


Takeaway: VIX often reduces the loss relative to price-only, but every representative absolute gate return remains negative.
